In [ ]:
import matplotlib.pyplot as plt
from statsmodels.base.model import GenericLikelihoodModel
import numpy as np
import cv2
import scienceplots
import tifffile as tiff

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia
from boulder_statistics.analysis.fit_params.power_law_fit_params import PowerLawFitParams
plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib inline

import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any
import tifffile
from typing import Dict
from typing import Callable


from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

dp = DataProductEncyclopedia(
    data_products_path=Path(r"C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\.data_products"))

In [ ]:
from boulder_statistics.analysis.quick_calculate_PowerLaw import PowerLawFittingFunction
from boulder_statistics.analysis.sensitivity_model import SensitivityModel

sm = SensitivityModel(dp.db_jaccard_agg)
fit = PowerLawFittingFunction(dp, LAD_min=0, sensitivity_model = SensitivityModel(dp.db_jaccard_agg))

In [ ]:
from statsmodels.base.model import GenericLikelihoodModelResults

mle_model: GenericLikelihoodModelResults = fit.MLE_fit(
    optimize_params = PowerLawFitParams(q=1, g=1.5),
    verbose = True
)

# LAD_min = 2 ; PowerLawFitParams(q=2, g=1.5)
# LAD_min = 4 ; PowerLawFitParams(q=1, g=1.3)

In [ ]:
# 1.6290 0.6006 for LAD min = 2
alphas_hist = np.geomspace(1, 1e6, 100)
alphas = np.geomspace(1, 1e6, 1000)

counts, bins, _ = plt.hist(fit.cleaned_data.collect()["alpha"].to_numpy(),
                           alphas_hist, density = True, label = rf"$LAD_{{\text{{min}}}} = {fit.LAD_min}$m")

fit_params = PowerLawFitParams(*mle_model.params)
# fit_params = PowerLawFitParams(q=0.9, g=0.8)
plt.plot(alphas, fit.F_norm(alphas, fit_params, sm.best_p_function), label = r"$F_{\text{PL}}$")

# plt.ylim(1e-6, counts.max() * 10)
# plt.xlim(fit.plot_range[0] * 0.7, fit.plot_range[1] * 0.5)
plt.xscale("log")
plt.xlabel(r"$\alpha$")
plt.yscale("log")
plt.ylabel("Det. probability density")
plt.legend()
plt.grid(which='both', linestyle='-', linewidth=0.4, alpha=0.2)
plt.tight_layout()
# plt.savefig(f".plots/PL_fit_LAS_gt_{fit.LAD_min}m.png")
plt.show()

In [ ]:
from datetime import datetime
from numpy.random import uniform
import scipy.interpolate

output_dir = Path("multi_LAD_min_fit_data")

init_g = scipy.interpolate.interp1d(
    [0, 2, 4],
    [2, 1.5, 1.3]
)

while True:
    LAD_min: float = uniform(0, 4)
    fit = PowerLawFittingFunction(dp, LAD_min=LAD_min, sensitivity_model = SensitivityModel(dp.db_jaccard_agg))

    df = fit.MultiMLEFit(
        optimize_params=PowerLawFitParams(q=1, g=init_g(LAD_min)),
        numb_runs=5,
        summary = False
    ).with_columns(
        pl.lit(LAD_min).alias("LAD_min")
    )

    # Filename based on current datetime (to the second)
    timestamp: str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename: Path = output_dir / f"{timestamp}.parquet"

    df.write_parquet(filename)

    print(f"Saved {filename}")

Optimization terminated successfully.
         Current function value: 8.473368
         Iterations: 35
         Function evaluations: 70


MultiMLE fit running:  60%|██████    | 3/5 [00:19<00:12,  6.45s/it]C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\src\boulder_statistics\analysis\quick_calculate_general.py:92: RuntimeWarning: divide by zero encountered in log
  np.log(total_p_alpha_sample),
C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\src\boulder_statistics\analysis\quick_calculate_general.py:149: RuntimeWarning: divide by zero encountered in log
  return np.log(F_norm(alphas, optimize_params, s_model))


Optimization terminated successfully.
         Current function value: 8.473074
         Iterations: 52
         Function evaluations: 97


MultiMLE fit running:  80%|████████  | 4/5 [00:27<00:07,  7.18s/it]C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\src\boulder_statistics\analysis\quick_calculate_general.py:92: RuntimeWarning: divide by zero encountered in log
  np.log(total_p_alpha_sample),
C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\src\boulder_statistics\analysis\quick_calculate_general.py:149: RuntimeWarning: divide by zero encountered in log
  return np.log(F_norm(alphas, optimize_params, s_model))


Optimization terminated successfully.
         Current function value: 8.473332
         Iterations: 29
         Function evaluations: 63


...

MultiMLE fit running: 100%|██████████| 5/5 [01:04<00:00, 12.99s/it]


Saved multi_LAD_min_fit_data\2026-06-30_11-14-10.parquet


MultiMLE fit running:   0%|          | 0/5 [00:00<?, ?it/s]C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\src\boulder_statistics\analysis\quick_calculate_general.py:92: RuntimeWarning: divide by zero encountered in log
  np.log(total_p_alpha_sample),
C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\src\boulder_statistics\analysis\quick_calculate_general.py:149: RuntimeWarning: divide by zero encountered in log
  return np.log(F_norm(alphas, optimize_params, s_model))


Optimization terminated successfully.
         Current function value: 9.480292
         Iterations: 43
         Function evaluations: 88


MultiMLE fit running:  20%|██        | 1/5 [00:08<00:32,  8.09s/it]C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\src\boulder_statistics\analysis\quick_calculate_general.py:92: RuntimeWarning: divide by zero encountered in log
  np.log(total_p_alpha_sample),
C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\src\boulder_statistics\analysis\quick_calculate_general.py:149: RuntimeWarning: divide by zero encountered in log
  return np.log(F_norm(alphas, optimize_params, s_model))
